In [ ]:
# import the system modules
import json
import datetime
import numpy as np
import pandas as pd
import random
import os
import importlib.util
import re

# Check if Faker is installed, if not, install it
if importlib.util.find_spec("faker") is None:
    !pip install Faker
from faker import Faker

# Mount the Google Drive if necessary
if importlib.util.find_spec("google.colab"):
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive")
    print("Google Drive mounted successfully.")
    print("Current working directory:", os.getcwd())

# import the CleanData class
from clean_data import CleanData
from pathlib import Path

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.1 MB/s eta 0:00:00


In [ ]:
data = CleanData("Output")

In [ ]:
data.print_filenames()

In [ ]:
def cleaning_fun(data_instance, **kwargs):
    # Totals
    fake = Faker()
    total_customers = 1000
    total_warehouses = 1000
    total_products = 36

    # maximums
    max_products_per_customer = kwargs.get("max_products_per_customer", 10)
    overwrite = kwargs.get("overwrite", False)

    # Discount probability parameters
    elite_discount_prob = kwargs.get("elite_discount_prob", 0.10)
    standard_discount_prob = kwargs.get("standard_discount_prob", 0.20)
    bargain_discount_prob = kwargs.get("bargain_discount_prob", 0.40)

    # Discount rate parameters (for when a discount *is* applied)
    elite_discount_rate = kwargs.get("elite_discount_rate", 0.15)
    standard_discount_rate = kwargs.get("standard_discount_rate", 0.30)
    bargain_discount_rate = kwargs.get("bargain_discount_rate", 0.45)

    # Return probability parameter
    return_probability = kwargs.get("return_probability", 0.02)

    # 2. --- SEASONALITY PREP ---
    # Normal months = 1, Holiday months = 2.5-4x more likely
    # Index 0=Jan, 11=Dec
    month_weights = [1.0, 1.0, 1.2, 1.1, 1.3, 1.2, 1.3, 1.4, 2.2, 2.5, 3.5, 4.5]
    months = list(range(1, 13))
    month_probs = [w/sum(month_weights) for w in month_weights]

    # 3. --- POOL GENERATION (Warehouses) ---
    warehouse_ids = random.sample(range(1000, 10000), total_warehouses)
    warehouse_pool = [{"WarehouseID": id,
                       "WarehouseName": f"Costco #{id}",
                       "WarehouseAddress": fake.street_address(),
                       "WarehouseCity": fake.city(),
                       "WarehouseArea": round(random.uniform(1000, 5000), 2),
                       "WarehouseZipCode": fake.zipcode(),
                       "WarehousePhoneNumber": fake.numerify("###-####")}
                       for id in warehouse_ids]

    # 4. --- POOL GENERATION (Products) ---
    # Define the Products per Department
    skus = random.sample(range(1000, 10000), total_products)
    dept_catalog = {
        12: ["Bulk Almonds", "Potato Chips", "Baking Flour", "Chocolate Bar", "Popcorn"],
        13: ["Bottled Water", "Apple Juice", "Soda 12-Pack", "Olive Oil", "Jasmine Rice"],
        18: ["Artisan Sourdough", "Croissants", "Bagels", "Dinner Rolls", "Fruit Pie"],
        19: ["Gourmet Cheese", "Gift Basket", "Pesto Sauce", "Truffle Oil"],
        23: ["AA Batteries", "Toaster", "LED Bulbs", "Headphones", "Coffee Maker"],
        25: ["Power Drill", "Automotive Oil", "Screwdriver Set", "Extension Cord"],
        61: ["Bath Towel Set", "Storage Bin", "Bed Sheets", "Kitchen Organizer"],
        65: ["Book", "Magazine Bundle", "Classic Movie DVD", "Gaming Headset"]
    }

    # Define Department Pricing Constraints (Min, Max)
    # This acts as our "Master Pricing Table"
    product_prices = {
        # Department 12
        "Bulk Almonds": (5.00, 12.00),
        "Potato Chips": (3.00, 6.00),
        "Baking Flour": (4.00, 8.00),
        "Chocolate Bar": (1.00, 3.00),
        "Popcorn": (3.00, 7.00),

        # Department 13
        "Bottled Water": (5.00, 10.00),
        "Apple Juice": (4.00, 9.00),
        "Soda 12-Pack": (6.00, 12.00),
        "Olive Oil": (10.00, 25.00),
        "Jasmine Rice": (5.00, 15.00),

        # Department 18
        "Artisan Sourdough": (4.00, 7.00),
        "Croissants": (5.00, 10.00),
        "Bagels": (3.00, 6.00),
        "Dinner Rolls": (3.00, 5.00),
        "Fruit Pie": (8.00, 15.00),

        # Department 19
        "Gourmet Cheese": (12.00, 30.00),
        "Gift Basket": (25.00, 60.00),
        "Pesto Sauce": (6.00, 12.00),
        "Truffle Oil": (15.00, 40.00),

        # Department 23
        "AA Batteries": (10.00, 20.00),
        "Toaster": (30.00, 80.00),
        "LED Bulbs": (8.00, 15.00),
        "Headphones": (40.00, 150.00),
        "Coffee Maker": (50.00, 150.00),

        # Department 25
        "Power Drill": (40.00, 120.00),
        "Automotive Oil": (20.00, 45.00),
        "Screwdriver Set": (15.00, 35.00),
        "Extension Cord": (10.00, 25.00),

        # Department 61
        "Bath Towel Set": (20.00, 50.00),
        "Storage Bin": (10.00, 30.00),
        "Bed Sheets": (30.00, 80.00),
        "Kitchen Organizer": (15.00, 40.00),

        # Department 65
        "Book": (10.00, 25.00),
        "Magazine Bundle": (5.00, 15.00),
        "Classic Movie DVD": (8.00, 20.00),
        "Gaming Headset": (30.00, 100.00)
    }

    # Create a flattened list of all unique products with their DeptID and Name
    all_products_details = []
    for d, names in dept_catalog.items():
        for n in names:
            all_products_details.append({"DeptID": d, "Name": n})

    # Combine unique SKUs with product details
    product_pool = []
    for i, product_detail in enumerate(all_products_details):
        min_p, max_p = product_prices.get(product_detail["Name"], (10.00, 20.00))
        product_pool.append({
            "SKU": skus[i],
            "DeptID": product_detail["DeptID"],
            "Name": product_detail["Name"],
            "BasePrice": round(random.uniform(min_p, max_p), 2)
        })

    # 5. --- POOL GENERATION (Discounted and Returned Products) ---
    num_products_to_discount = random.randint(7, 14)
    num_products_to_return = random.randint(7, 11)
    discounted_product_skus = random.sample(skus, k=num_products_to_discount)
    returned_product_skus = random.sample(skus, k=num_products_to_return)

    # 6. --- OUTPUT MANAGEMENT ---
    # Static function counter variable
    if not hasattr(cleaning_fun, "counter"):
        cleaning_fun.counter = 1
    else:
        cleaning_fun.counter += 1

    # Output file name extraction
    _, output_path = data_instance.extract_filenames()
    while not overwrite and os.path.exists(output_path):
        base, ext = os.path.splitext(output_path)
        output_path = f"{base} ({cleaning_fun.counter}){ext}"

    # 7. --- MAIN SIMULATION LOOP ---
    # Initialization
    records = []
    customer_ids = random.sample(range(1000, 10000), total_customers)
    global_transaction_id_counter = 1000000
    global_register_id_counter = 1000
    customers_using_discounts = set()
    customers_returning_products = set()
    products_being_discounted = set()
    products_being_returned = set()

    # Looping Through the customer ids and appending records
    print(f"Generating {total_customers} tiered customers with {total_products} products...")

    for id in customer_ids:
        # --- CUSTOMER PROFILE ---
        # Determine Tier
        roll = random.random()
        if roll < 0.20:
            tier = "ELITE"
            min_d, max_d = 7, 14
        elif roll < 0.70:
            tier = "STANDARD"
            min_d, max_d = 14, 30
        else:
            tier = "BARGAIN"
            min_d, max_d = 30, 60

        # Profile Dictionary
        cust_profile = {
            "CustomerID": id,
            "CustomerTier": tier,
            "CustomerFirstName": fake.first_name(),
            "CustomerLastName": fake.last_name(),
            "CustomerAddress": fake.street_address(),
            "CustomerCity": fake.city(),
            "CustomerState": fake.state_abbr(),
            "CustomerZipCode": fake.zipcode(),
            "CustomerPhoneNumber": fake.numerify("###-####")
        }

        # Instead of uniform random, weight the START dates by your seasonality
        start_month = random.choices(months, weights=month_weights, k=1)[0]
        start_year = 2025 # User specified

        try:
            current_date = datetime.date(start_year, start_month, random.randint(1, 28))
        except ValueError: # Handle cases where day might be out of range for a month
            current_date = datetime.date(start_year, start_month, 1) # Default to 1st of month

        # Ensure current_date is not in the future relative to datetime.date.today()
        # This handles cases where start_month/day might push it slightly past today within the same year
        if current_date > datetime.date.today():
            current_date = datetime.date.today()

        current_transaction_date = current_date

        # To prevent the "Exhaustion" cliff, ensure trips can continue
        # up until the actual current date of the simulation.

        # Calculate actual number of trips possible based on duration to today
        days_since_customer_start = (datetime.date.today() - current_date).days

        # Determine average trip interval based on tier's min_d and max_d
        avg_trip_interval = (min_d + max_d) / 2

        if days_since_customer_start >= 0 and avg_trip_interval > 0:
            # Estimate trips based on total available days and average interval
            estimated_trips_from_start = days_since_customer_start / avg_trip_interval
            # Add some randomness (e.g., 70% to 130% of estimated trips), ensuring at least one
            num_trips_base = max(1, int(estimated_trips_from_start * random.uniform(0.7, 1.3)))
        else:
            num_trips_base = 1 # Ensure at least one trip even if start date is today or calculation is zero


        for _ in range(num_trips_base):
            # -- TRANSACTION PROFILE ---
            # Transaction and Return dates
            current_transaction_date += datetime.timedelta(days=random.randint(min_d, max_d))
            current_transaction_time = int(fake.time(pattern="%H%M"))
            return_days = datetime.timedelta(days=random.randint(1, 30)) # 30-Day return policy
            current_return_date = current_transaction_date+return_days
            current_return_time = int(fake.time(pattern="%H%M"))

            # Transaction and Register IDs
            current_transaction_id = global_transaction_id_counter
            global_transaction_id_counter += 1
            current_register_id = global_register_id_counter
            global_register_id_counter += 1

            # Profile Dictionary
            transaction_profile = {
                "TransactionID": current_transaction_id,
                "TransactionDate": current_transaction_date,
                "RegisterID": current_register_id,
                "Time": current_transaction_time
            }

            # --- WAREHOUSE PROFILE ---
            # Pick a warehouse from our pre-generated pool
            warehouse = random.choice(warehouse_pool)

            # Profile Dictionary
            warehouse_profile = {
                "WarehouseID": warehouse["WarehouseID"],
                "WarehouseName": warehouse["WarehouseName"],
                "WarehouseAddress": warehouse["WarehouseAddress"],
                "WarehouseCity": warehouse["WarehouseCity"],
                "WarehouseArea": warehouse["WarehouseArea"],
                "WarehouseZipCode": warehouse["WarehouseZipCode"],
                "WarehousePhoneNumber": warehouse["WarehousePhoneNumber"]
            }

            # Black Friday Spike (November 24-30 range)
            current_transaction_month = current_transaction_date.month
            current_transaction_day = current_transaction_date.day
            if current_transaction_month == 11 and \
               current_transaction_day >= 24 and current_transaction_day <= 30:
                is_black_friday_week = True
            else:
                is_black_friday_week = False

            # Transaction Volume: Buy more items during Holidays/Black Friday
            if is_black_friday_week:
                vol_boost = 2
            elif current_transaction_month >= 10:
                vol_boost = 1.5
            else:
                vol_boost = 1
            num_products_to_buy = int(random.randint(1, max_products_per_customer)*vol_boost)
            num_products_to_select = min(num_products_to_buy, len(product_pool))

            # Returned Products Data
            selected_products = random.sample(product_pool, k=num_products_to_select)
            product_list = []

            for prod in selected_products:
                # --- PRODUCT PROFILE ---
                # Apply noise + holiday pricing pressure
                noise = random.uniform(-0.5, 2.00)
                unit_price_before_discount = round(prod["BasePrice"] + noise, 2)

                # Quantity: Weighted heavily for bulk buys during spikes
                if is_black_friday_week:
                    quantities, q_weights = [1, 5, 10, 20], [0.10, 0.30, 0.40, 0.20]
                else:
                    quantities, q_weights = [1, 2, 3, 4, 5, 10, 20], [0.50, 0.25, 0.15, 0.05, 0.03, 0.01, 0.01]
                quantity = random.choices(quantities, weights=q_weights, k=1)[0]

                # Profile Dictionary
                product_profile = {
                    "SKU": prod["SKU"],
                    "DepartmentID": prod["DeptID"],
                    "ProductDescription": prod["Name"].upper(),
                    "BasePrice": prod["BasePrice"]
                }

                # --- DISCOUNT LOGIC ---
                # Initialize for zero discount assumption
                applied_discount_rate = 0
                apply_discount_this_time = False

                # Determine discount rate based on customer tier
                if prod["SKU"] in discounted_product_skus:
                    if tier == "ELITE":
                        if random.random() < elite_discount_prob:
                            applied_discount_rate = elite_discount_rate
                            apply_discount_this_time = True
                    elif tier == "STANDARD":
                        if random.random() < standard_discount_prob:
                            applied_discount_rate = standard_discount_rate
                            apply_discount_this_time = True
                    else: # tier == "BARGAIN"
                        if random.random() < bargain_discount_prob:
                            applied_discount_rate = bargain_discount_rate
                            apply_discount_this_time = True

                # Apply the discount if applicable
                if apply_discount_this_time and applied_discount_rate > 0:
                    total_discount_for_item = -round(quantity * unit_price_before_discount * applied_discount_rate, 2)
                    customers_using_discounts.add(id)
                    products_being_discounted.add(prod["SKU"])
                else:
                    total_discount_for_item = 0

                # --- PRIMARY RECORD ---
                primary_record = {
                    **transaction_profile,
                    **cust_profile,
                    **warehouse_profile,
                    **product_profile,
                    "Quantity": quantity,
                    "OriginalUnitPrice": unit_price_before_discount,
                    "DiscountAmount": total_discount_for_item,
                    "TaxCode": random.choice(["3", "Y"])
                }
                records.append(primary_record)
                product_list.append(primary_record)

            # --- RETURN LOGIC ---
            # Copy the primary records
            return_record = [p.copy() for p in product_list]

            # Add return records
            for r in return_record:
                if r["SKU"] in returned_product_skus and \
                   random.random() < return_probability:
                    # Set return quantity to negative
                    quantities = [1, 2, 3, 4, 5, 10, 20]
                    q_weights = [0.50, 0.25, 0.15, 0.05, 0.03, 0.01, 0.01]
                    return_quantity = -random.choices(quantities, weights=q_weights, k=1)[0]

                    # Determine the unit price for the returned item
                    # For simplicity, use the base price with some noise
                    noise = random.uniform(-0.5, 2.00)
                    unit_price_at_return = round(r["BasePrice"]+noise, 2)

                    # Construct the return record using details from the returned_product dictionary
                    r["TransactionDate"] = current_return_date
                    r["Time"] = current_return_time
                    r["Quantity"] = return_quantity
                    r["OriginalUnitPrice"] = unit_price_at_return
                    r["DiscountAmount"] = 0

                    # Append the return record
                    records.append(r)
                    customers_returning_products.add(r["CustomerID"])
                    products_being_returned.add(r["SKU"])

    # 8. --- CREATE AND EXPORT ---
    data_frame = pd.DataFrame(records)
    data_frame.to_csv(output_path, index=False)

    print(f"\nFile exported to: {output_path}")
    print(f"Total Rows: {len(data_frame)}")
    print(f"Peak Transaction Month: {data_frame['TransactionDate'].apply(lambda x: x.strftime('%B')).mode()[0]}")
    print(f"Total Unique TransactionIDs generated: {data_frame["TransactionID"].nunique()}")
    print(f"Total Unique CustomerIDs generaged: {data_frame["CustomerID"].nunique()}")
    print(f"Total Unique SKUs generaged: {data_frame["SKU"].nunique()}")
    print(f"Total Unique Customers Using Discounts: {len(customers_using_discounts)}")
    print(f"Total Unique Discounted Products: {len(products_being_discounted)}")
    print(f"Total Unique Customers Returning Products: {len(customers_returning_products)}")
    print(f"Total Unique Returned Products: {len(products_being_returned)}\n")

    return data_frame

In [ ]:
df = data.process_files(cleaning_fun)